In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database('F1_ANALYTICS')
session.use_schema('GOLD')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# -------------------------------
# LOAD DATA
# -------------------------------
df = session.table('F1_ANALYTICS.GOLD.QUALIFICATION_RESULTS').to_pandas()

# -------------------------------
# SELECT FEATURES (SIMPLIFIED)
# -------------------------------
feature_cols = ['AVG_QUALI_TIME', 'FASTEST_QUALIFYING_TIME']
df = df.dropna(subset=feature_cols)

X = df[feature_cols]

# -------------------------------
# SCALE FEATURES
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# K-MEANS CLUSTERING
# -------------------------------
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# -------------------------------
# PERFORMANCE SCORE
# -------------------------------
df['performance_score'] = 1 / df['AVG_QUALI_TIME']

# -------------------------------
# MAP CLUSTERS → WIN PROBABILITY
# -------------------------------
cluster_perf = df.groupby('cluster')['performance_score'].mean()
sorted_clusters = cluster_perf.sort_values(ascending=False)

cluster_mapping = {
    sorted_clusters.index[0]: 'Best Chance',
    sorted_clusters.index[1]: 'High Chance',
    sorted_clusters.index[2]: 'Moderate Chance',
    sorted_clusters.index[3]: 'Low Chance',
    sorted_clusters.index[4]: 'Very Low Chance'
    
}

df['win_category'] = df['cluster'].map(cluster_mapping)

# -------------------------------
# OPTIONAL VALIDATION
# -------------------------------
if 'POSITION' in df.columns:
    df['winner_flag'] = (df['POSITION'] == 1).astype(int)
    print(df.groupby('win_category')['winner_flag'].mean())

# -------------------------------
# VISUALIZATION (DIRECT FEATURES)
# -------------------------------
plt.figure(figsize=(10, 6))

for label in df['win_category'].unique():
    subset = df[df['win_category'] == label]
    plt.scatter(
        subset['AVG_QUALI_TIME'],
        subset['FASTEST_QUALIFYING_TIME'],
        label=label
    )
    # Add driver_number labels
    for _, row in subset.iterrows():
        plt.text(
            row['AVG_QUALI_TIME'],
            row['FASTEST_QUALIFYING_TIME'],
            str(row['DRIVER_NUMBER']),
            fontsize=9,
            ha='right',   # horizontal alignment
            va='bottom'   # vertical alignment
        )

plt.xlabel("Average Qualifying Time")
plt.ylabel("Fastest Qualifying Time")
plt.title("Driver Clusters (Simple Model)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score
n_clusters =  5
# -------------------------------
# SILHOUETTE SCORE
# -------------------------------
if n_clusters > 1:  # silhouette score requires at least 2 clusters
    score = silhouette_score(X_scaled, df['cluster'])
    print(f"Silhouette Score: {score:.3f}")
else:
    print("Silhouette score not applicable for a single cluster.")

In [ ]:
from snowflake.snowpark import Session

# Assume you already have a Snowpark session
snowpark_df = session.create_dataframe(selected_df)

snowpark_df.write.save_as_table(
    "F1_ANALYTICS.GOLD.CLUSTERING_RESULTS",
    mode="overwrite"
)
